# Density Matrix Analysis: Fermi Hubbard

### PAULI-BASIS MEASUREMENTS AND DATA COLLECTION

In [ ]:
import numpy as np
from numpy import pi
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

shots = 

total_time = 
iterations = 
dt = total_time / iterations

#Fermi Hubbard Parameters
t =       # hopping strength
u =       # onsite interaction

print("Backend:", backend.name)
print("dt per Trotter step:", dt)

def fermihubbard_circuit(n, t, u, dt):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)

    for _ in range(n):

        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(-2 * t * dt, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)

        qc.sdg(0)
        qc.sdg(1)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(-2 * t * dt, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(0)
        qc.s(1)

        qc.cx(0, 1)
        qc.rz(-2 * u * dt, 1)
        qc.cx(0, 1)

    return qc

# BASIS ROTATIONS + MEASUREMENT
def add_basis_and_measure(qc, basis):
    qc2 = qc.copy()
    for q, b in enumerate(basis):
        if b == "X":
            qc2.h(q)
        elif b == "Y":
            qc2.sdg(q)
            qc2.h(q)
    qc2.measure_all()
    return qc2

bases = ["ZZ","ZX","ZY","ZI",
         "XZ","XX","XY","XI",
         "YZ","YX","YY","YI",
         "IZ","IX","IY"]

all_circuits = []

for n in range(1, iterations + 1):
    for basis in bases:
        base = fermihubbard_circuit(n, t, u, dt)
        full = add_basis_and_measure(base, basis)
        all_circuits.append(full)

print("Total circuits submitted:", len(all_circuits)) #Total 75 Circuits

# TRANSPILATION + EXECUTION
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(all_circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=shots)

print("\nSAVE THIS JOB ID:")
print(job.job_id())

### RECONSTRUCTION OF DENSITY MATRIX: THE TOMOGRAPHY ANALYSIS USING REAL HARDWARE RESULTS

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.quantum_info import DensityMatrix, Pauli
from qiskit.visualization import plot_state_city

# ================================
# LOAD JOB
# ================================
service = QiskitRuntimeService(channel="ibm_quantum_platform")
job = service.job("d6jdl9cmmeis739rcheg")
result = job.result()

print("Job loaded")
print("Total circuits:", len(result))

# ================================
# FULL TOMOGRAPHY BASES (15)
# ================================
bases = [
    "ZZ","ZX","ZY","ZI",
    "XZ","XX","XY","XI",
    "YZ","YX","YY","YI",
    "IZ","IX","IY"
]

# ================================
# SAFE COUNTS EXTRACTOR
# ================================
def get_counts(pub):
    try:
        return pub.data.meas.get_counts()
    except:
        bitstrings = pub.data.meas.get_bitstrings()
        counts = {}
        for b in bitstrings:
            counts[b] = counts.get(b, 0) + 1
        return counts

# ================================
# EXPECTATION VALUE
# ================================
def expectation_from_counts(counts):
    shots = sum(counts.values())
    val = 0.0
    for bit, c in counts.items():
        parity = (-1)**(bit.count("1"))
        val += parity * c / shots
    return val

# ================================
# RECONSTRUCT RHO
# ================================
def reconstruct_density_matrix(result, iteration):
    start = iteration * len(bases)
    expvals = {}

    for i, basis in enumerate(bases):
        counts = get_counts(result._pub_results[start + i])
        expvals[basis] = expectation_from_counts(counts)

    rho = np.zeros((4,4), dtype=complex)

    for a in "IXYZ":
        for b in "IXYZ":
            if a == "I" and b == "I":
                coeff = 1.0
            else:
                coeff = expvals.get(a + b, 0.0)
            rho += coeff * Pauli(a + b).to_matrix()

    return rho / 4

# ================================
# DETECT ITERATIONS
# ================================
num_steps = len(result) // len(bases)
print("Detected iterations:", num_steps)

# ================================
# PLOT
# ================================
for step in range(num_steps):
    rho = reconstruct_density_matrix(result, step)

    print(f"\nIteration {step+1} density matrix:\n")
    print(np.round(rho, 3))

    fig = plot_state_city(DensityMatrix(rho))
    display(fig)

    if step == 0: fig.savefig("iteration_1fermi_density_matrix.png", dpi=300)
    if step == 4: fig.savefig("iteration_5fermi_density_matrix.png", dpi=300)
    
    plt.close(fig)

### COMPARISON OF TOMOGRAPHY ANALYSIS RESULTS: HARDWARE VS IDEAL USING HEAT MAPS

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.quantum_info import DensityMatrix, Pauli

# =====================================
# LOAD JOB FROM IBM
# =====================================
service = QiskitRuntimeService(channel="ibm_quantum_platform")
job = service.job("d6jdl9cmmeis739rcheg")   # <-- your job ID
result = job.result()

print("Job loaded")
print("Total circuits:", len(result))

# =====================================
# FULL 2-QUBIT TOMOGRAPHY BASES
# =====================================
bases = [
    "ZZ","ZX","ZY","ZI",
    "XZ","XX","XY","XI",
    "YZ","YX","YY","YI",
    "IZ","IX","IY"
]

# =====================================
# SAFE COUNTS EXTRACTOR
# =====================================
def get_counts(pub):
    try:
        return pub.data.meas.get_counts()
    except:
        bitstrings = pub.data.meas.get_bitstrings()
        counts = {}
        for b in bitstrings:
            counts[b] = counts.get(b, 0) + 1
        return counts

# =====================================
# EXPECTATION VALUE FROM COUNTS
# =====================================
def expectation_from_counts(counts):
    shots = sum(counts.values())
    value = 0.0
    for bit, count in counts.items():
        parity = (-1) ** (bit.count("1"))
        value += parity * count / shots
    return value

# =====================================
# RECONSTRUCT DENSITY MATRIX
# =====================================
def reconstruct_density_matrix(result, iteration):
    start = iteration * len(bases)
    expvals = {}

    for i, basis in enumerate(bases):
        counts = get_counts(result._pub_results[start + i])
        expvals[basis] = expectation_from_counts(counts)

    rho = np.zeros((4, 4), dtype=complex)

    for a in "IXYZ":
        for b in "IXYZ":
            if a == "I" and b == "I":
                coeff = 1.0
            else:
                coeff = expvals.get(a + b, 0.0)
            rho += coeff * Pauli(a + b).to_matrix()

    rho = rho / 4

    # Make matrix numerically clean
    rho = (rho + rho.conj().T) / 2
    rho = rho / np.trace(rho)

    return rho

# =====================================
# DETECT NUMBER OF ITERATIONS
# =====================================
num_steps = len(result) // len(bases)
print("Detected iterations:", num_steps)

hardware_density_matrix = []

# =====================================
# RECONSTRUCT ALL DENSITY MATRICES
# =====================================
for step in range(num_steps):
    rho = reconstruct_density_matrix(result, step)
    hardware_density_matrix.append(rho)

    print(f"\nIteration {step+1} density matrix:\n")
    print(np.round(rho, 3))

print("\nAll density matrices stored.")

# =====================================
# HEATMAP FUNCTION
# =====================================
def plot_density_heatmaps(rho_a, rho_b,
                          title_a="Iteration 1",
                          title_b="Iteration 5"):

    basis_labels = ['00', '01', '10', '11']

    fig, axs = plt.subplots(2, 3, figsize=(15, 8))

    # ----- REAL PART -----
    sns.heatmap(np.real(rho_a), annot=True, fmt=".2f",
                cmap='coolwarm',
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 15},   # inside numbers
                ax=axs[0, 0])
    axs[0, 0].set_title(f"{title_a} - Real", fontsize=17)

    sns.heatmap(np.real(rho_b), annot=True, fmt=".2f",
                cmap='coolwarm',
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 12},
                ax=axs[0, 1])
    axs[0, 1].set_title(f"{title_b} - Real", fontsize=14)

    sns.heatmap(np.real(rho_b - rho_a), annot=True, fmt=".2f",
                cmap='bwr', center=0,
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 13.5},
                ax=axs[0, 2])
    axs[0, 2].set_title("Difference - Real", fontsize=12.8)

    # ----- IMAGINARY PART -----
    sns.heatmap(np.imag(rho_a), annot=True, fmt=".2f",
                cmap='coolwarm',
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 12},
                ax=axs[1, 0])
    axs[1, 0].set_title(f"{title_a} - Imag", fontsize=14)

    sns.heatmap(np.imag(rho_b), annot=True, fmt=".2f",
                cmap='coolwarm',
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 13.5},
                ax=axs[1, 1])
    axs[1, 1].set_title(f"{title_b} - Imag", fontsize=14)

    sns.heatmap(np.imag(rho_b - rho_a), annot=True, fmt=".2f",
                cmap='bwr', center=0,
                xticklabels=basis_labels,
                yticklabels=basis_labels,
                annot_kws={"size": 13.5},
                ax=axs[1, 2])
    axs[1, 2].set_title("Difference - Imag", fontsize=14)

    # Increase axis + colorbar fonts
    for ax in axs.flat:
        ax.tick_params(axis='both', labelsize=12.5)
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=12.5)

    plt.tight_layout()
    fig.savefig("density_matrix_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()

rho_1 = hardware_density_matrix[0]
rho_5 = hardware_density_matrix[4]

plot_density_heatmaps(rho_1, rho_5,"Iteration 1", "Iteration 5")